In [ ]:
import pandas as pd

transactions = pd.read_parquet('data/credit_card_transactions-ibm_v2.parquet')
cards = pd.read_parquet('data/sd254_cards.parquet')
users = pd.read_parquet('data/sd254_users.parquet')


In [ ]:
cards

In [ ]:
users

In [ ]:
transactions

In [ ]:
users['User'] = users.index

data = transactions.merge(users, on='User', how='left')
cards = cards.rename(columns={'CARD INDEX': 'Card'})

all_data = data.merge(cards, on=['User', 'Card'], how='left')

all_data

pasiskaiciuojam kiek procentu fraud

In [ ]:
mask= all_data['Is Fraud?'] == "Yes"
fraud_transactions = all_data.loc[mask]
len_fraud = len(fraud_transactions)
len_non_fraud = len(all_data) - len_fraud
print("Fraud transactions:", len_fraud)
print("Non fraud transactions:", len_non_fraud)

print("percentage of fraud transactions:", len_fraud / len(all_data) * 100)

tiriam tik online transakcijas

In [ ]:
mask = all_data["Merchant City"] == 'ONLINE'
data = all_data.loc[mask].copy()
data

pasiziurim kiek procentu fraud yra online transakcijpse

In [ ]:
mask= data['Is Fraud?'] == "Yes"
fraud_transactions = data.loc[mask]
len_fraud = len(fraud_transactions)
len_non_fraud = len(data) - len_fraud
print("Fraud transactions:", len_fraud)
print("Non fraud transactions:", len_non_fraud)

print("percentage of fraud transactions:", len_fraud / len(data) * 100)

pasiziurim kiek none  values turim transakcijose

In [ ]:
missing = data.isna().sum()
print(missing)

kadangi Merchant State ir Zip neturi duomenu, o apartment neturi reiksmes, trinam situos stulpelius lauk

In [ ]:
data.drop(columns = ['Merchant State', 'Zip', 'Apartment'], inplace = True)

In [ ]:
data

In [ ]:
import matplotlib.pyplot as plt

fraud = data[data['Is Fraud?'] == 'Yes'].copy()

fraud.groupby('Year')['Is Fraud?'].count().plot(kind='bar')
plt.title('Fraud pagal metus')
plt.show()

fraud.groupby('Month')['Is Fraud?'].count().plot(kind='bar')
plt.title('Fraud pagal menesi')
plt.show()

fraud.groupby('Day')['Is Fraud?'].count().plot(kind='bar')
plt.title('Fraud pagal savaitės dieną')
plt.show()



kadangi nenesa svarbios info, trinam:
Use Chip
Merchant City
Person
Current Age
Birth Month
Address
City
State
Latitude
Longitude
Card Number
Expires
CVV
Has Chip
Cards Issued
Acct Open Date
Year PIN last Changed
Card on Dark Web (visi no)
Time
Retirement Age
MCC
Merchant Name
Credit Limit
Card
User
Zipcode
Per Capita Income - Zipcode

reikia encodint: gender, card brand, card type

In [ ]:
data

In [ ]:
cols_to_drop = '''
Use Chip
Merchant City
Person
Current Age
Birth Month
Address
City
State
Latitude
Longitude
Card Number
Expires
CVV
Has Chip
Cards Issued
Acct Open Date
Year PIN last Changed
Card on Dark Web
Time
Retirement Age
MCC
Merchant Name
Credit Limit
Card
User
Zipcode
Per Capita Income - Zipcode
'''.strip().splitlines()

cols_to_drop = [col.strip() for col in cols_to_drop]

data.drop(columns=cols_to_drop, inplace=True)

In [ ]:
data

In [ ]:
mask = data["Errors?"].isna()
data.loc[mask, "Errors?"] = False
data.loc[~mask, "Errors?"] = True
data

In [ ]:
un = transactions["Errors?"].unique()
print(un)

In [ ]:
mask = ~(data['Errors?'].isna()) & (data['Is Fraud?'] == 'Yes')
test = data.loc[mask].copy()
test


In [ ]:
mask = ~(data['Errors?'].isna()) & (data['Is Fraud?'] == 'No')
test = data.loc[mask].copy()
test


In [ ]:
from decimal import Decimal

data['Yearly Income - Person'] = data['Yearly Income - Person'].str.replace('$', '', regex=False).apply(Decimal)
data['Total Debt'] = data['Total Debt'].str.replace('$', '', regex=False).apply(Decimal)
data['Amount'] = data['Amount'].str.replace('$', '', regex=False).apply(Decimal)


In [ ]:
data

In [ ]:
mask = data['Gender'] == "Female"
data.loc[mask, 'Gender'] = 1

mask = data['Gender'] == "Male"
data.loc[mask, 'Gender'] = 2


In [ ]:
mask = data['Card Brand'] == "Visa"
data.loc[mask, 'Card Brand'] = 1

mask = data['Card Brand'] == "Mastercard"
data.loc[mask, 'Card Brand'] = 2

mask = data['Card Brand'] == "Amex"
data.loc[mask, 'Card Brand'] = 3

mask = data['Card Brand'] == "Discover"
data.loc[mask, 'Card Brand'] = 4


In [ ]:
mask = data['Card Type'] == "Debit"
data.loc[mask, 'Card Type'] = 1

mask = data['Card Type'] == "Credit"
data.loc[mask, 'Card Type'] = 2

mask = data['Card Type'] == "Debit (Prepaid)"
data.loc[mask, 'Card Type'] = 3


In [ ]:
mask = data['Errors?'] == True
data.loc[mask, 'Errors?'] = 1

mask = data['Errors?'] == False
data.loc[mask, 'Errors?'] = 0

In [ ]:
mask = data['Is Fraud?'] == "Yes"
data.loc[mask, 'Is Fraud?'] = 1

mask = data['Is Fraud?'] == "No"
data.loc[mask, 'Is Fraud?'] = 0

In [ ]:
data

In [ ]:
for col in data.columns:
    if data[col].dtype == 'object':
        try:
            data[col] = pd.to_numeric(data[col], errors='coerce')
            print(f"{col} konvertuota")
        except:
            print(f"{col} nekonvertuota")

In [ ]:
data

In [ ]:
data['Age'] = data['Year'] - data['Birth Year']

In [ ]:
data

In [ ]:
data.to_parquet('data/prepared_data.parquet', index=False)